# 🔍 Deteksi Anomali Ekonomi Makro — DBSCAN

**Early Warning System Krisis Ekonomi menggunakan DBSCAN (Spatial Clustering)**

---

Notebook ini mengimplementasikan pendekatan **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) untuk mendeteksi anomali pada indikator ekonomi makro sebagai sistem peringatan dini (*Early Warning System*) krisis ekonomi.

**Metodologi:**
- Prinsip: Clustering berbasis densitas — titik-titik *noise* (label = -1) dianggap sebagai anomali
- Grid Search: Variasi `eps` × variasi `min_samples` untuk mencari parameter optimal
- Evaluasi: Precision, Recall, F1-Score, ROC-AUC terhadap *ground truth* krisis historis
- Interpretasi: Analisis fitur per cluster dan identifikasi pola anomali

**Dataset:** 14 indikator makroekonomi, 49 negara, periode 1990-2024 (World Bank Open Data)  
**Disusun oleh:** Bram

## 1. Instalasi & Import Library

In [ ]:
# Import Library
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, average_precision_score,
    precision_recall_curve, roc_auc_score,
    silhouette_score
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

SEED = 42
np.random.seed(SEED)

print('Library berhasil diimport')
import sklearn; print(f'scikit-learn: {sklearn.__version__}')

## 2. Load & Eksplorasi Data

In [ ]:
df = pd.read_csv('../data_cleaned.csv')

print(f'Shape dataset: {df.shape}')
print(f'Jumlah negara: {df["economy"].nunique()}')
print(f'Rentang tahun: {df["year"].min()} - {df["year"].max()}')
print(f'\nDistribusi crisis_label:')
print(df['crisis_label'].value_counts())
print(f'\nPersentase krisis: {df["crisis_label"].mean()*100:.2f}%')

df.head()

In [ ]:
FEATURE_COLS = [
    'GDP_Growth', 'GDP_PerCapita_Growth', 'Inflation_CPI',
    'Total_Reserves', 'Unemployment', 'Current_Account_GDP',
    'Trade_GDP', 'FDI_Inflows_GDP', 'Exports_GDP', 'Imports_GDP',
    'Gross_Savings_GDP', 'Exchange_Rate', 'Manufacturing_Value',
    'Investment_GDP'
]

FEATURE_LABELS = {
    'GDP_Growth': 'Pertumbuhan PDB (%)',
    'GDP_PerCapita_Growth': 'PDB per Kapita Growth (%)',
    'Inflation_CPI': 'Inflasi CPI (%)',
    'Total_Reserves': 'Cadangan Devisa',
    'Unemployment': 'Pengangguran (%)',
    'Current_Account_GDP': 'Neraca Berjalan/PDB (%)',
    'Trade_GDP': 'Perdagangan/PDB (%)',
    'FDI_Inflows_GDP': 'FDI Masuk/PDB (%)',
    'Exports_GDP': 'Ekspor/PDB (%)',
    'Imports_GDP': 'Impor/PDB (%)',
    'Gross_Savings_GDP': 'Tabungan Bruto/PDB (%)',
    'Exchange_Rate': 'Nilai Tukar (LCU/USD)',
    'Manufacturing_Value': 'Manufaktur/PDB (%)',
    'Investment_GDP': 'Investasi/PDB (%)'
}

print(f'Jumlah fitur: {len(FEATURE_COLS)}')
print(f'Missing values per fitur:')
print(df[FEATURE_COLS].isnull().sum())
print(f'\nStatistik deskriptif:')
df[FEATURE_COLS].describe().round(3)

In [ ]:
# Visualisasi distribusi fitur
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    
    # Histogram normal vs krisis
    normal = df[df['crisis_label'] == 0][col]
    crisis = df[df['crisis_label'] == 1][col]
    
    ax.hist(normal, bins=40, alpha=0.6, label='Normal', color='steelblue', density=True)
    ax.hist(crisis, bins=40, alpha=0.6, label='Krisis', color='tomato', density=True)
    
    label = FEATURE_LABELS.get(col, col)
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=7)

# Sembunyikan sumbu kosong
for j in range(len(FEATURE_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Fitur: Normal vs Krisis', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 3. Analisis K-Distance untuk Estimasi Epsilon

Sebelum melakukan grid search, kita melakukan analisis k-distance plot untuk mendapatkan estimasi awal parameter `eps` yang baik.

In [ ]:
# Ekstraksi fitur
X = df[FEATURE_COLS].values
y_true = df['crisis_label'].values

# K-distance plot untuk berbagai nilai k (min_samples)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, k in enumerate([5, 10, 15]):
    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors_fit = neighbors.fit(X)
    distances, indices = neighbors_fit.kneighbors(X)
    
    # Ambil jarak ke tetangga ke-k, urutkan
    k_distances = np.sort(distances[:, k-1])
    
    ax = axes[idx]
    ax.plot(range(len(k_distances)), k_distances, color='steelblue', linewidth=1)
    ax.set_title(f'K-Distance Plot (k={k})', fontsize=12)
    ax.set_xlabel('Data Points (sorted)')
    ax.set_ylabel(f'{k}-th Nearest Neighbor Distance')
    ax.grid(True, alpha=0.3)
    
    # Tandai area "elbow" — estimasi eps
    # Cari titik elbow menggunakan perbedaan gradien
    gradients = np.gradient(k_distances)
    elbow_idx = np.argmax(gradients > np.mean(gradients) + 2 * np.std(gradients))
    if elbow_idx > 0:
        ax.axhline(y=k_distances[elbow_idx], color='red', linestyle='--', alpha=0.7,
                   label=f'Estimasi eps ≈ {k_distances[elbow_idx]:.2f}')
        ax.legend()

plt.suptitle('Analisis K-Distance untuk Estimasi Parameter eps', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Ground Truth Krisis Historis

Definisi krisis historis yang digunakan sebagai *ground truth* untuk evaluasi performa model.

In [ ]:
# Ground truth krisis historis
GROUND_TRUTH = {
    'Krisis Asia 1997-1998': {
        'tahun': [1997, 1998],
        'negara': ['IDN', 'THA', 'MYS', 'KOR', 'PHL']
    },
    'Krisis Rusia 1998': {
        'tahun': [1998],
        'negara': ['RUS']
    },
    'Krisis Argentina 2001-2002': {
        'tahun': [2001, 2002],
        'negara': ['ARG']
    },
    'Global Financial Crisis 2008-2009': {
        'tahun': [2008, 2009],
        'negara': list(df['economy'].unique())
    },
    'Krisis Utang Eropa 2010-2012': {
        'tahun': [2010, 2011, 2012],
        'negara': ['GRC', 'PRT', 'IRL', 'ESP', 'ITA']
    },
    'Pandemi COVID-19 2020': {
        'tahun': [2020],
        'negara': list(df['economy'].unique())
    }
}

print('Ground Truth Krisis Historis:')
print(f'{"Krisis":<40} {"Jumlah Observasi":>20}')
print('-' * 62)

total_crisis = 0
for nama, info in GROUND_TRUTH.items():
    mask = df['economy'].isin(info['negara']) & df['year'].isin(info['tahun'])
    count = mask.sum()
    total_crisis += count
    print(f'{nama:<40} {count:>20}')

print('-' * 62)
print(f'{"Total observasi krisis (crisis_label=1)":<40} {df["crisis_label"].sum():>20}')
print(f'{"Total observasi":<40} {len(df):>20}')
print(f'{"Persentase krisis":<40} {df["crisis_label"].mean()*100:>19.2f}%')

## 5. Grid Search — Optimasi Parameter DBSCAN

DBSCAN memiliki dua parameter utama:
- **`eps`** (epsilon): Radius neighbourhood — menentukan jarak maksimum antar dua titik agar masih dianggap satu cluster
- **`min_samples`**: Jumlah minimum titik dalam radius `eps` agar sebuah titik dianggap sebagai *core point*

Titik yang tidak masuk ke cluster manapun dilabeli sebagai **noise (-1)** dan dianggap sebagai **anomali/krisis**.

In [ ]:
# ── Grid Search DBSCAN ─────────────────────────────────────────────────
# Target: ~8% observasi terdeteksi sebagai anomali (sesuai rata-rata frekuensi krisis historis)

eps_values = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
min_samples_values = [3, 5, 7, 10, 15]

results = []

for eps in eps_values:
    for min_samples in min_samples_values:
        # Fit DBSCAN
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean', n_jobs=-1)
        labels = dbscan.fit_predict(X)
        
        # Noise (-1) = anomali
        y_pred = (labels == -1).astype(int)
        
        # Jumlah cluster dan noise
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        pct_noise = n_noise / len(labels) * 100
        
        # Skip jika tidak ada anomali terdeteksi atau semua dianggap anomali
        if n_noise == 0 or n_noise == len(labels):
            continue
        
        # Metrics
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        results.append({
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'pct_noise': pct_noise,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })

df_results = pd.DataFrame(results)

if len(df_results) > 0:
    print(f'Total kombinasi yang menghasilkan anomali: {len(df_results)}')
    print(f'\nTop 10 berdasarkan F1-Score:')
    display(df_results.sort_values('f1_score', ascending=False).head(10).round(4))
else:
    print('Tidak ada kombinasi parameter yang menghasilkan anomali. Silakan perluas range parameter.')

In [ ]:
# Visualisasi hasil grid search
if len(df_results) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # F1-Score heatmap
    pivot_f1 = df_results.pivot_table(values='f1_score', index='min_samples', columns='eps', aggfunc='first')
    sns.heatmap(pivot_f1, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0])
    axes[0].set_title('F1-Score', fontsize=13)
    
    # Precision heatmap
    pivot_prec = df_results.pivot_table(values='precision', index='min_samples', columns='eps', aggfunc='first')
    sns.heatmap(pivot_prec, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1])
    axes[1].set_title('Precision', fontsize=13)
    
    # Recall heatmap
    pivot_recall = df_results.pivot_table(values='recall', index='min_samples', columns='eps', aggfunc='first')
    sns.heatmap(pivot_recall, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[2])
    axes[2].set_title('Recall', fontsize=13)
    
    plt.suptitle('Grid Search DBSCAN — Heatmap Metrik Evaluasi', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Visualisasi persentase noise
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    pivot_noise = df_results.pivot_table(values='pct_noise', index='min_samples', columns='eps', aggfunc='first')
    sns.heatmap(pivot_noise, annot=True, fmt='.1f', cmap='Blues', ax=ax)
    ax.set_title('Persentase Noise/Anomali (%)', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Simpan hasil grid search
if len(df_results) > 0:
    df_results.sort_values('f1_score', ascending=False).to_csv('grid_search_dbscan.csv', index=False)
    print('Hasil grid search disimpan ke: grid_search_dbscan.csv')

## 6. Model Terbaik — DBSCAN

Melatih model DBSCAN dengan parameter terbaik dari hasil grid search.

In [ ]:
# Pilih parameter terbaik berdasarkan F1-Score
if len(df_results) > 0:
    best = df_results.sort_values('f1_score', ascending=False).iloc[0]
    best_eps = best['eps']
    best_min_samples = int(best['min_samples'])
else:
    # Default jika grid search gagal
    best_eps = 3.0
    best_min_samples = 5

print(f'Parameter terbaik:')
print(f'  eps = {best_eps}')
print(f'  min_samples = {best_min_samples}')

# Fit model terbaik
best_dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples, metric='euclidean', n_jobs=-1)
best_labels = best_dbscan.fit_predict(X)

# Anomali = noise (-1)
y_pred = (best_labels == -1).astype(int)

n_clusters = len(set(best_labels)) - (1 if -1 in best_labels else 0)
n_noise = (best_labels == -1).sum()

print(f'\nHasil DBSCAN:')
print(f'  Jumlah cluster: {n_clusters}')
print(f'  Jumlah noise/anomali: {n_noise} ({n_noise/len(best_labels)*100:.1f}%)')
print(f'  Jumlah titik dalam cluster: {(best_labels != -1).sum()}')
print(f'\nDistribusi cluster:')
for label in sorted(set(best_labels)):
    count = (best_labels == label).sum()
    label_name = 'Noise/Anomali' if label == -1 else f'Cluster {label}'
    print(f'  {label_name}: {count} ({count/len(best_labels)*100:.1f}%)')

## 7. Evaluasi Model

In [ ]:
# Classification report
print('=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomali/Krisis'], zero_division=0))

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f'\nMetrik Ringkasan:')
print(f'  Precision: {precision:.4f}')
print(f'  Recall: {recall:.4f}')
print(f'  F1-Score: {f1:.4f}')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix angka
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomali'],
            yticklabels=['Normal', 'Krisis'],
            ax=axes[0])
axes[0].set_title('Confusion Matrix (Angka)', fontsize=13)
axes[0].set_xlabel('Prediksi DBSCAN')
axes[0].set_ylabel('Ground Truth')

# Confusion matrix persentase
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=['Normal', 'Anomali'],
            yticklabels=['Normal', 'Krisis'],
            ax=axes[1])
axes[1].set_title('Confusion Matrix (Persentase %)', fontsize=13)
axes[1].set_xlabel('Prediksi DBSCAN')
axes[1].set_ylabel('Ground Truth')

plt.suptitle(f'DBSCAN (eps={best_eps}, min_samples={best_min_samples})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Visualisasi — PCA & t-SNE

In [ ]:
# PCA 2D
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# PCA — Ground Truth
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true,
                           cmap='coolwarm', alpha=0.6, s=15, edgecolors='none')
axes[0].set_title('PCA — Ground Truth', fontsize=13)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter1, ax=axes[0], label='0=Normal, 1=Krisis')

# PCA — Prediksi DBSCAN
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_pred,
                           cmap='coolwarm', alpha=0.6, s=15, edgecolors='none')
axes[1].set_title('PCA — Prediksi DBSCAN', fontsize=13)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter2, ax=axes[1], label='0=Normal, 1=Anomali')

# PCA — Cluster labels
scatter3 = axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=best_labels,
                           cmap='tab20', alpha=0.6, s=15, edgecolors='none')
axes[2].set_title('PCA — Cluster DBSCAN', fontsize=13)
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[2].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter3, ax=axes[2], label='Cluster ID (-1=Noise)')

plt.suptitle('Proyeksi PCA 2D — DBSCAN', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f'Explained variance ratio: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total explained variance: {sum(pca.explained_variance_ratio_)*100:.1f}%')

In [ ]:
# t-SNE 2D
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# t-SNE — Ground Truth
scatter1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_true,
                           cmap='coolwarm', alpha=0.6, s=15, edgecolors='none')
axes[0].set_title('t-SNE — Ground Truth', fontsize=13)
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
plt.colorbar(scatter1, ax=axes[0], label='0=Normal, 1=Krisis')

# t-SNE — Prediksi DBSCAN
scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_pred,
                           cmap='coolwarm', alpha=0.6, s=15, edgecolors='none')
axes[1].set_title('t-SNE — Prediksi DBSCAN', fontsize=13)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
plt.colorbar(scatter2, ax=axes[1], label='0=Normal, 1=Anomali')

# t-SNE — Cluster labels
scatter3 = axes[2].scatter(X_tsne[:, 0], X_tsne[:, 1], c=best_labels,
                           cmap='tab20', alpha=0.6, s=15, edgecolors='none')
axes[2].set_title('t-SNE — Cluster DBSCAN', fontsize=13)
axes[2].set_xlabel('t-SNE 1')
axes[2].set_ylabel('t-SNE 2')
plt.colorbar(scatter3, ax=axes[2], label='Cluster ID (-1=Noise)')

plt.suptitle('Proyeksi t-SNE 2D — DBSCAN', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 9. Visualisasi Temporal — Timeline Deteksi Anomali

In [ ]:
# Timeline anomali per tahun
df_eval = df[['economy', 'year', 'crisis_label']].copy()
df_eval['dbscan_anomaly'] = y_pred

# Agregasi per tahun
yearly_gt = df_eval.groupby('year')['crisis_label'].sum()
yearly_pred = df_eval.groupby('year')['dbscan_anomaly'].sum()

fig, ax = plt.subplots(figsize=(16, 6))

years = yearly_gt.index
width = 0.35

ax.bar(years - width/2, yearly_gt.values, width, label='Ground Truth (Krisis)', color='tomato', alpha=0.7)
ax.bar(years + width/2, yearly_pred.values, width, label='DBSCAN (Anomali)', color='steelblue', alpha=0.7)

# Tandai periode krisis
crisis_years = [1997, 1998, 2001, 2002, 2008, 2009, 2010, 2011, 2012, 2020]
for cy in crisis_years:
    ax.axvline(x=cy, color='gray', linestyle=':', alpha=0.4)

ax.set_xlabel('Tahun', fontsize=12)
ax.set_ylabel('Jumlah Negara Terdeteksi', fontsize=12)
ax.set_title('Timeline Deteksi Anomali DBSCAN vs Ground Truth Krisis Historis', fontsize=14)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap deteksi per negara per tahun
pivot_pred = df_eval.pivot_table(values='dbscan_anomaly', index='economy', columns='year', aggfunc='first')
pivot_gt = df_eval.pivot_table(values='crisis_label', index='economy', columns='year', aggfunc='first')

fig, axes = plt.subplots(2, 1, figsize=(20, 16))

sns.heatmap(pivot_gt, cmap='Reds', ax=axes[0], cbar_kws={'label': '1=Krisis'},
            linewidths=0.1, linecolor='white')
axes[0].set_title('Ground Truth — Krisis Historis', fontsize=13)
axes[0].set_xlabel('Tahun')
axes[0].set_ylabel('Negara')

sns.heatmap(pivot_pred, cmap='Blues', ax=axes[1], cbar_kws={'label': '1=Anomali'},
            linewidths=0.1, linecolor='white')
axes[1].set_title('Prediksi DBSCAN — Anomali Terdeteksi', fontsize=13)
axes[1].set_xlabel('Tahun')
axes[1].set_ylabel('Negara')

plt.suptitle(f'Perbandingan Ground Truth vs DBSCAN (eps={best_eps}, min_samples={best_min_samples})',
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 10. Analisis Feature Importance

Mengidentifikasi fitur yang paling berkontribusi dalam membedakan titik anomali (noise) dari titik normal (cluster member) menggunakan pendekatan *mean difference*.

In [ ]:
# Feature importance berdasarkan perbedaan rata-rata fitur antara anomali dan normal
df_fi = df[FEATURE_COLS].copy()
df_fi['anomaly'] = y_pred

normal_mean = df_fi[df_fi['anomaly'] == 0][FEATURE_COLS].mean()
anomaly_mean = df_fi[df_fi['anomaly'] == 1][FEATURE_COLS].mean()

# Perbedaan absolut rata-rata
mean_diff = (anomaly_mean - normal_mean).abs()
mean_diff_sorted = mean_diff.sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart feature importance
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(mean_diff_sorted)))
bars = axes[0].barh(range(len(mean_diff_sorted)),
                    mean_diff_sorted.values,
                    color=colors)
axes[0].set_yticks(range(len(mean_diff_sorted)))
labels = [FEATURE_LABELS.get(f, f) for f in mean_diff_sorted.index]
axes[0].set_yticklabels(labels, fontsize=9)
axes[0].set_xlabel('|Mean Anomali - Mean Normal|', fontsize=11)
axes[0].set_title('Feature Importance (Mean Difference)', fontsize=13)

# Perbandingan rata-rata per fitur
top_features = mean_diff.sort_values(ascending=False).head(7).index.tolist()
comparison_data = pd.DataFrame({
    'Fitur': [FEATURE_LABELS.get(f, f) for f in top_features],
    'Normal': [normal_mean[f] for f in top_features],
    'Anomali': [anomaly_mean[f] for f in top_features]
})

x = np.arange(len(top_features))
width = 0.35
axes[1].bar(x - width/2, comparison_data['Normal'], width, label='Normal', color='steelblue', alpha=0.7)
axes[1].bar(x + width/2, comparison_data['Anomali'], width, label='Anomali', color='tomato', alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison_data['Fitur'], rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Rata-rata Nilai (standardized)', fontsize=11)
axes[1].set_title('Top 7 Fitur — Normal vs Anomali', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Analisis Feature Importance — DBSCAN', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 11. Analisis per Krisis Historis

In [ ]:
# Evaluasi per krisis historis
print('=' * 80)
print('EVALUASI DBSCAN PER KRISIS HISTORIS')
print('=' * 80)

crisis_analysis = []

for nama, info in GROUND_TRUTH.items():
    mask = df['economy'].isin(info['negara']) & df['year'].isin(info['tahun'])
    crisis_obs = df_eval[mask]
    
    if len(crisis_obs) == 0:
        continue
    
    n_total = len(crisis_obs)
    n_detected = crisis_obs['dbscan_anomaly'].sum()
    detection_rate = n_detected / n_total * 100 if n_total > 0 else 0
    
    crisis_analysis.append({
        'Krisis': nama,
        'Total_Observasi': n_total,
        'Terdeteksi': n_detected,
        'Detection_Rate': detection_rate
    })
    
    print(f'\n--- {nama} ---')
    print(f'  Total observasi: {n_total}')
    print(f'  Terdeteksi DBSCAN: {n_detected} ({detection_rate:.1f}%)')
    
    # Detail per negara
    if n_detected > 0:
        detected_countries = crisis_obs[crisis_obs['dbscan_anomaly'] == 1][['economy', 'year']]
        print(f'  Negara terdeteksi:')
        for _, row in detected_countries.iterrows():
            print(f'    - {row["economy"]} ({row["year"]})')

print('\n' + '=' * 80)

# Ringkasan
df_crisis_analysis = pd.DataFrame(crisis_analysis)
if len(df_crisis_analysis) > 0:
    print('\nRingkasan Detection Rate per Krisis:')
    display(df_crisis_analysis.round(1))

In [ ]:
# Visualisasi detection rate per krisis
if len(df_crisis_analysis) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    bars = ax.bar(range(len(df_crisis_analysis)), 
                  df_crisis_analysis['Detection_Rate'],
                  color=plt.cm.Set2(np.linspace(0, 1, len(df_crisis_analysis))))
    
    ax.set_xticks(range(len(df_crisis_analysis)))
    ax.set_xticklabels(df_crisis_analysis['Krisis'], rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Detection Rate (%)', fontsize=12)
    ax.set_title('DBSCAN — Detection Rate per Krisis Historis', fontsize=14)
    ax.set_ylim(0, 105)
    
    # Tambahkan label nilai
    for bar, val in zip(bars, df_crisis_analysis['Detection_Rate']):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
    
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50%')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 12. Simpan Hasil

In [ ]:
# Simpan hasil deteksi lengkap
df_output = df[['economy', 'year', 'crisis_label']].copy()
df_output['dbscan_anomaly'] = y_pred
df_output['dbscan_cluster'] = best_labels

# Tambahkan semua fitur
for col in FEATURE_COLS:
    df_output[col] = df[col]

df_output.to_csv('hasil_dbscan.csv', index=False)
print(f'Hasil deteksi disimpan ke: hasil_dbscan.csv')
print(f'Shape: {df_output.shape}')

# Simpan ringkasan model
summary = pd.DataFrame([{
    'model': 'DBSCAN',
    'eps': best_eps,
    'min_samples': best_min_samples,
    'n_clusters': n_clusters,
    'n_anomalies': n_noise,
    'pct_anomalies': round(n_noise / len(y_pred) * 100, 2),
    'precision': round(precision, 4),
    'recall': round(recall, 4),
    'f1_score': round(f1, 4)
}])
summary.to_csv('ringkasan_model_dbscan.csv', index=False)
print(f'\nRingkasan model disimpan ke: ringkasan_model_dbscan.csv')
display(summary)

## 13. Ringkasan & Kesimpulan

### Metodologi DBSCAN
- **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) mengidentifikasi cluster berdasarkan kepadatan (*density*) titik data dalam ruang fitur dimensi tinggi.
- Titik yang tidak termasuk dalam cluster manapun dilabeli sebagai **noise (-1)** dan diperlakukan sebagai **anomali/krisis**.
- Pendekatan ini sangat cocok untuk data ekonomi makro karena:
  1. Tidak memerlukan asumsi bentuk cluster tertentu
  2. Secara natural mengidentifikasi outlier sebagai noise
  3. Mampu menangkap pola non-linear dan interaksi kompleks antar indikator

### Parameter Optimal
- Parameter dipilih melalui grid search yang mengoptimalkan F1-Score terhadap ground truth krisis historis
- Target: ~8% observasi terdeteksi sebagai anomali (sesuai rata-rata frekuensi krisis historis)

### Evaluasi
- Model dievaluasi terhadap 6 periode krisis historis:
  1. Krisis Asia 1997-1998
  2. Krisis Rusia 1998
  3. Krisis Argentina 2001-2002
  4. Global Financial Crisis 2008-2009
  5. Krisis Utang Eropa 2010-2012
  6. Pandemi COVID-19 2020

### Kekuatan DBSCAN
- Tidak memerlukan jumlah cluster *a priori*
- Secara otomatis mengidentifikasi titik-titik yang "terasing" dari formasi cluster utama
- Robust terhadap perbedaan ukuran dan bentuk cluster

### Keterbatasan
- Sensitif terhadap pemilihan parameter `eps` dan `min_samples`
- Kurang efektif pada data dengan variasi densitas yang sangat tinggi
- Tidak menghasilkan skor kontinu (hanya label biner: cluster vs noise)